# EUR-Lex AI Chat — GPU Embedding

Embeds 2.2M chunks from `chunks.db` on T4 GPU, builds FAISS index, uploads to HF Hub.

**Expected time:** ~30 minutes on T4 GPU (free Colab)

---

## Setup

1. Go to **Runtime → Change runtime type** → Select **T4 GPU**
2. Click the 🔑 **Secrets** button (left panel) → Add your `HF_TOKEN`
3. Click **Runtime → Run all**

In [ ]:
# @title 1. Install dependencies
import sys, time, os, json, sqlite3
from datetime import UTC, datetime

!pip install -q sentence-transformers huggingface_hub tqdm psutil

# Try to import faiss (pre-installed on Colab), install if missing
try:
    import faiss
    print(f"FAISS already installed (version {faiss.__version__})")
except ImportError:
    print("Installing faiss-cpu...")
    !pip install -q faiss-cpu
    import faiss
    print(f"FAISS installed (version {faiss.__version__})")

print("\nAll dependencies ready!")

In [ ]:
# @title 2. Authenticate and configure
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret not found!")

api = HfApi()
who = api.whoami(token=HF_TOKEN)["name"]
print(f"Logged in as: {who}")

REPO_ID = f"{who}/eurlex-chat-data"
chunks_remote_path = "chunks_raw.db"

create_repo(REPO_ID, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
print(f"Dataset repo: {REPO_ID}")

In [ ]:
# @title 3. Download chunks_raw.db from HuggingFace Hub
from huggingface_hub import hf_hub_download
import shutil

CHUNKS_DB = "chunks_raw.db"

if not os.path.exists(CHUNKS_DB):
    print("Downloading chunks_raw.db from HF Hub...")
    start = time.time()
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=chunks_remote_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    shutil.copy(path, CHUNKS_DB)
    elapsed = time.time() - start
    size_gb = os.path.getsize(CHUNKS_DB) / 1e9
    print(f"Downloaded {size_gb:.1f} GB in {elapsed/60:.1f} min")
else:
    print(f"Already cached: {CHUNKS_DB}")

conn = sqlite3.connect(CHUNKS_DB)
count = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
celexes = conn.execute("SELECT COUNT(DISTINCT celex) FROM chunks").fetchone()[0]
conn.close()
print(f"Chunks: {count:,}, Documents: {celexes:,}")

In [ ]:
# @title 4. Load model and embed all chunks on GPU
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm.notebook import tqdm

MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 256

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device="cuda")
print(f"Model loaded on: {model.device}")

conn = sqlite3.connect(CHUNKS_DB)
total = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"Total chunks to embed: {total:,}")

cursor = conn.execute("SELECT text FROM chunks ORDER BY id")

all_embeddings = []
pbar = tqdm(total=total, desc="Embedding", unit="chunks")

embed_start = time.time()
while True:
    rows = cursor.fetchmany(BATCH_SIZE)
    if not rows:
        break
    texts = [r[0] for r in rows]
    emb = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    all_embeddings.append(emb)
    pbar.update(len(texts))

conn.close()
pbar.close()

vectors = np.vstack(all_embeddings).astype(np.float32)
del all_embeddings

embed_elapsed = time.time() - embed_start
print(f"\nEmbedding complete: {vectors.shape}")
print(f"Time: {embed_elapsed/60:.1f} min ({total/embed_elapsed:.0f} chunks/s)")

In [ ]:
# @title 5. Build FAISS index (IVFPQ)
import faiss

n_vectors, dim = vectors.shape
n_centroids = min(int(4 * np.sqrt(n_vectors)), max(n_vectors // 40, 1))
n_centroids = max(n_centroids, 1)

print(f"Building FAISS IVF{n_centroids}+PQ48x8 index...")
print(f"Vector data size: {vectors.nbytes/1e9:.1f} GB")

index = faiss.index_factory(dim, f"IVF{n_centroids},PQ48x8", faiss.METRIC_INNER_PRODUCT)

print("Training...")
index.train(vectors)

print("Adding vectors...")
index.add(vectors)
del vectors

index.use_precomputed_table = -1
index.nprobe = min(50, n_centroids)

INDEX_PATH = "index.faiss"
faiss.write_index(index, INDEX_PATH)
size_mb = os.path.getsize(INDEX_PATH) / 1e6
print(f"Index saved: {size_mb:.1f} MB")
print(f"Vectors: {index.ntotal}, Dimension: {index.d}")

In [ ]:
# @title 6. Upload results to HuggingFace Hub
INDEX_PATH = "index.faiss"
CHUNKS_DB = "chunks_raw.db"

ts = datetime.now(UTC).isoformat()
build_meta = {
    "timestamp": ts,
    "n_chunks": index.ntotal,
    "n_documents": celexes,
    "model": MODEL_NAME,
    "dimension": 384,
    "index_type": "IVFPQ",
    "source": "colab-gpu-embedding",
}

print("Uploading to HF Hub...")
upload_start = time.time()

api.upload_file(
    repo_id=REPO_ID,
    path_in_repo="index.faiss",
    path_or_fileobj=INDEX_PATH,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"  Uploaded index.faiss ({os.path.getsize(INDEX_PATH)/1e6:.0f} MB)")

api.upload_file(
    repo_id=REPO_ID,
    path_in_repo="chunks.db",
    path_or_fileobj=CHUNKS_DB,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"  Uploaded chunks.db ({os.path.getsize(CHUNKS_DB)/1e9:.1f} GB)")

with open("last_updated.txt", "w") as f:
    f.write(ts)
api.upload_file(
    repo_id=REPO_ID,
    path_in_repo="last_updated.txt",
    path_or_fileobj="last_updated.txt",
    repo_type="dataset",
    token=HF_TOKEN,
)

with open("build_meta.json", "w") as f:
    json.dump(build_meta, f, indent=2)
api.upload_file(
    repo_id=REPO_ID,
    path_in_repo="build_meta.json",
    path_or_fileobj="build_meta.json",
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"Upload complete! ({time.time()-upload_start:.0f}s)")
print(f"\nAll files on HuggingFace Hub at: {REPO_ID}")

---
## Done

Come back to your terminal and tell me it finished. I'll handle the rest.